# Lekcja 11 - Agent-do-agenta (A2A) Protokół


## Konfiguracja


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Co to jest protokół A2A?

The **Protokół Agent-do-Agenta (A2A)** to otwarty standard, który umożliwia agentom AI komunikację,
odnajdywanie się nawzajem i współpracę — nawet gdy są zbudowane na różnych frameworkach lub hostowane
przez różne usługi.

Kluczowe pojęcia:

- **Discovery** – Agenci publikują *Kartę Agenta*, która opisuje ich możliwości, ułatwiając
  innym agentom (lub koordynatorom) znalezienie odpowiedniego specjalisty do zadania.
- **Message Passing** – Agenci wymieniają się ustrukturyzowanymi wiadomościami poprzez wspólny protokół, dzięki czemu
  żądanie od jednego agenta może być zrozumiane i zrealizowane przez innego niezależnie od jego wewnętrznej
  implementacji.
- **Task Lifecycle** – A2A definiuje stany takie jak *submitted*, *working*, *completed*, oraz
  *failed*, dając koordynatorowi pełen wgląd w to, jak postępuje przekazane zadanie.

W tej lekcji symulujemy współpracę w stylu A2A, łącząc trzy wyspecjalizowane agenty turystyczne
w przepływ pracy, w którym każdy agent wnosi swoją ekspertyzę i przekazuje wyniki następnemu.


## Tworzenie wyspecjalizowanych agentów podróży


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Współpraca wielu agentów poprzez przepływ pracy

Łączymy trzy agenty w sekwencyjny przepływ pracy, który odzwierciedla przekazywanie wiadomości A2A:

1. **CurrencyExchangeAgent** otrzymuje żądanie użytkownika i przygotowuje wskazówki dotyczące waluty.
2. **ActivityPlannerAgent** otrzymuje wzbogacony kontekst i dodaje rekomendacje aktywności.
3. **TravelManagerAgent** łączy oba wejścia w końcowy raport podróżny.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Understanding A2A in Production

In a production environment the A2A protocol unlocks powerful cross-service scenarios:

| Możliwość | Opis |
|---|---|
| **Interoperacyjność między frameworkami** | Agent zbudowany przy użyciu jednego frameworka może delegować zadania agentowi zbudowanemu przy użyciu dowolnego innego frameworka zgodnego z A2A, umożliwiając prawdziwą interoperacyjność między organizacjami. |
| **Granice usług** | Agenci mogą działać w oddzielnych mikroserwisach, regionach chmurowych, a nawet w różnych organizacjach, nadal współpracując bez zakłóceń. |
| **Dynamiczne odkrywanie** | Orkiestrator może zapytać rejestr Agent Card w czasie wykonywania, aby znaleźć najlepiej dopasowanego specjalistę do danego podzadania. |
| **Strumieniowanie i powiadomienia push** | A2A obsługuje Server-Sent Events (SSE) dla aktualizacji postępu w czasie rzeczywistym oraz powiadomień push dla zadań długotrwałych. |

The workflow we built above is a simplified, in-process version of this pattern. W rzeczywistej
instalacji każdy agent wystawiałby punkt końcowy HTTP, publikował Agent Card i komunikował
za pomocą protokołu A2A JSON-RPC.


## Podsumowanie

W tej lekcji dowiedziałeś się:

1. **Czym jest protokół A2A** — otwarty standard do wykrywania agentów, przesyłania wiadomości i zarządzania zadaniami.
2. **Jak tworzyć wyspecjalizowanych agentów** — agenta Currency Exchange, agenta Activity Planner i orkiestratora Travel Manager.
3. **Jak podłączyć agentów do przepływu pracy** — używając `WorkflowBuilder` do modelowania sekwencyjnego przesyłania wiadomości między agentami.
4. **Jak A2A działa w produkcji** — umożliwiając współpracę między różnymi frameworkami i usługami dzięki dynamicznemu wykrywaniu i strumieniowym aktualizacjom.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Zastrzeżenie:
Niniejszy dokument został przetłumaczony przy użyciu usługi tłumaczeniowej AI Co-op Translator (https://github.com/Azure/co-op-translator). Chociaż dokładamy starań, aby tłumaczenia były jak najdokładniejsze, prosimy pamiętać, że tłumaczenia automatyczne mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku źródłowym należy traktować jako źródło nadrzędne. W przypadku informacji o znaczeniu krytycznym zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
